# 🌲 Sparse AI: Topological Needle-In-A-Haystack (NIAH) Benchmark Sweep
### Empirical Retrieval Retention Curve & Phase Transition ("Retrieval Knee") Across Context Length $N = 4096$

This notebook runs the swept benchmark to close the loop on selective KV routing in **Sparse AI** on the primary paper model (**Meta-Llama-3.1-8B-Instruct**) or **TinyLlama-1.1B**:
1. **Control Variable**: Sweeps required matching depth $r \in [0, 1, 2, 3, 4, 5]$ across the Bruhat-Tits tree, corresponding to theoretical activation sparsities $0\%, 50\%, 75\%, 87.5\%, 93.8\%, 96.9\%$ (formally verified in `PrefixSparsity.lean`).
2. **Effective Cache Capacities**: Evaluates equivalent memory footprints $K \in [4096, 2048, 1024, 512, 256, 128]$.
3. **Metric**: Needle retrieval recall rate across **50 random positions** (early, middle, late context).
4. **Expected Curve**: Replaces isolated extreme compression failures with the standard selective KV phase transition: **>95% recall** when active budget > 20%, a sharp **retrieval knee** at ~12–15%, and **information starvation** below 10%.

In [ ]:
# Step 1: Hardware Check & Hugging Face Secrets Authentication
import os
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
gpu_name = "CPU"
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Device: {gpu_name}")
    print(f"Total VRAM: {vram_gb:.2f} GB")

# Load HF_TOKEN from Colab Secrets
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ["HF_TOKEN"] = token
        print("✅ HF_TOKEN successfully authenticated from Colab Secrets.")
    else:
        print("ℹ️ HF_TOKEN not set in Colab Secrets; using public access.")
except Exception as e:
    print(f"Running in local/standard environment: {e}")

In [ ]:
# Step 2: Clone/Pull Repository & Install Dependencies
import os
from pathlib import Path

if not Path("src/llama_surgery").exists():
    if Path("sparse-ai").exists():
        %cd sparse-ai
        !git pull origin main
    else:
        !git clone https://github.com/sneed-and-feed/sparse-ai.git
        %cd sparse-ai
else:
    !git pull origin main

# Install dependencies (including bitsandbytes for 8-bit support on T4)
!pip install -q transformers accelerate bitsandbytes matplotlib seaborn tqdm
!pip install -q -e .

In [ ]:
# Step 3: Run the Swept Topological NIAH Benchmark
# Choose your model configuration below:

# OPTION A: Meta-Llama-3.1-8B-Instruct on A100 / L4 GPU (Full bfloat16, ~45-60 min)
# !python experiments/sweep_topological_niah.py \
#     --model_id "meta-llama/Meta-Llama-3.1-8B-Instruct" \
#     --context_len 4096 \
#     --num_positions 50 \
#     --train_steps 80

# OPTION B: Meta-Llama-3.1-8B-Instruct on free Tesla T4 (8-bit Quantized, ~35 min)
# !python experiments/sweep_topological_niah.py \
#     --model_id "meta-llama/Meta-Llama-3.1-8B-Instruct" \
#     --load_in_8bit \
#     --context_len 4096 \
#     --num_positions 50 \
#     --train_steps 80

# OPTION C: TinyLlama-1.1B (Fastest testbed, ~15 min on free T4)
!python experiments/sweep_topological_niah.py \
    --model_id "meta-llama/Meta-Llama-3.1-8B-Instruct" \
    --context_len 4096 \
    --num_positions 50 \
    --train_steps 80

In [ ]:
# Step 4: Display the Empirical Retrieval Curves & Matrix
from IPython.display import Image, display
import json

print("===================================================================")
print("FIGURE 1: RETRIEVAL RETENTION VS. ACTIVE KV BUDGET (THE KNEE)")
print("===================================================================")
if Path("figures/retrieval_knee_curve.png").exists():
    display(Image("figures/retrieval_knee_curve.png"))
else:
    print("Figure not found. Check benchmark execution logs.")

print("\n===================================================================")
print("FIGURE 2: PASS/FAIL RETRIEVAL MATRIX (CONTEXT POSITION VS. DEPTH)")
print("===================================================================")
if Path("figures/retrieval_heatmap.png").exists():
    display(Image("figures/retrieval_heatmap.png"))

In [ ]:
# Step 5: Print Summary Metrics Table for BENCHMARKS.md
results_file = Path("experiments/results/niah_sweep_results.json")
if results_file.exists():
    with open(results_file, "r") as f:
        data = json.load(f)
    
    print("| Matching Depth $r$ | Active Budget (%) | Theoretical Sparsity | Effective Cache $K$ | Retrieval Recall (%) |")
    print("| :---: | :---: | :---: | :---: | :---: |")
    for k, v in data["sweep"].items():
        print(f"| $r={v['r']}$ | {v['active_budget_pct']}% | {v['sparsity_pct']}% | {v['effective_capacity_k']} | **{v['recall_pct']}%** ({v['successes']}/{v['total']}) |")
else:
    print("No results file found.")